# 06 · Decision Tree

**What this notebook covers**

- Grid search over depth, leaf size and split criterion
- Overfitting check: tree depth vs train and CV score
- Feature importance
- A shallow 3-level tree with readable rules (text and diagram)

## 1. Environment setup
### 1.1 Imports
Standard model-notebook imports plus the estimator-specific ones.

In [ ]:
import hashlib
import json
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score, brier_score_loss, f1_score, make_scorer,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

### 1.2 Configuration
Every tunable knob lives here. Lower the search budget for a quick dry run, but use
the defaults for the reported results.

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_PREFIX = "06"
NOTEBOOK_NAME = "06_decision_tree.ipynb"
CV_FOLDS = 5

VARIANTS = ["day_zero", "full_lifecycle"]
TREE_GRID = {
    "model__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, None],
    "model__min_samples_leaf": [1, 10, 50, 100],
    "model__criterion": ["gini", "entropy"],
}
EXPLAINER_DEPTH = 3

### 1.3 Project paths, shared constants and helpers

In [ ]:
# Resolve the repository root regardless of where the kernel was started
# (VS Code starts kernels in notebooks/, Jupyter may start them at the repo root).
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "DECISIONS.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (no DECISIONS.md in any parent directory).")


PROJECT_ROOT = find_project_root()

PATHS = {
    "raw": PROJECT_ROOT / "data" / "hotel_bookings.csv",
    "clean": PROJECT_ROOT / "data" / "interim" / "hotel_bookings_clean.csv",
    "splits": PROJECT_ROOT / "data" / "processed" / "splits",
    "features": PROJECT_ROOT / "data" / "processed" / "features",
    "artifacts": PROJECT_ROOT / "artifacts",
    "preprocessors": PROJECT_ROOT / "artifacts" / "preprocessors",
    "model_outputs": PROJECT_ROOT / "artifacts" / "model_outputs",
    "models": PROJECT_ROOT / "models",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "tables": PROJECT_ROOT / "reports" / "tables",
    "notebooks": PROJECT_ROOT / "notebooks",
}
for key in ("artifacts", "preprocessors", "model_outputs", "models", "figures", "tables"):
    PATHS[key].mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Columns that must always be treated as categorical strings when re-reading CSVs.
STRING_COLUMNS = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "agent", "company",
    "customer_type", "arrival_season",
]
TARGET = "is_canceled"
ID_COLUMN = "booking_id"
LEAKAGE_COLUMNS = ["reservation_status", "reservation_status_date"]

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Streaming SHA-256 of a file (used to fingerprint every artifact)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path.relative_to(PROJECT_ROOT)}")
    return json.loads(path.read_text())


def read_frame(path: Path) -> pd.DataFrame:
    """Read a project CSV with categorical columns forced to plain Python strings.

    `keep_default_na=False` + an explicit NA list stops pandas from turning legitimate
    category labels into NaN; numeric columns are parsed normally.
    """
    if not path.exists():
        raise FileNotFoundError(f"Required data file missing: {path.relative_to(PROJECT_ROOT)}")
    frame = pd.read_csv(
        path,
        dtype={col: object for col in STRING_COLUMNS},
        keep_default_na=False,
        na_values=["", "NaN", "nan"],
    )
    present = [c for c in STRING_COLUMNS if c in frame.columns]
    frame[present] = frame[present].astype(object)
    return frame

In [ ]:
# One visual language for every figure in the project:
#  - blue = not cancelled, orange = cancelled (validated colour-blind-safe pair)
#  - single-series charts use blue; recessive grey grid; no top/right spines; no dual axes.
COLORS = {
    "not_canceled": "#2a78d6",
    "canceled": "#eb6834",
    "neutral": "#8a8985",
    "ink": "#0b0b0b",
    "ink_muted": "#52514e",
}
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQUENTIAL_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_diverging", ["#2a78d6", "#e8e7e3", "#eb6834"]
)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "axes.prop_cycle": mpl.cycler(color=SERIES_COLORS),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.labelcolor": COLORS["ink_muted"],
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.titlecolor": COLORS["ink"],
    "axes.grid": True,
    "grid.color": "#e8e7e3",
    "grid.linewidth": 0.8,
    "xtick.color": COLORS["ink_muted"],
    "ytick.color": COLORS["ink_muted"],
    "lines.linewidth": 2,
    "legend.frameon": False,
})


def save_figure(fig, name: str) -> Path:
    """Save a figure under reports/figures with this notebook's numeric prefix."""
    path = PATHS["figures"] / f"{NOTEBOOK_PREFIX}_{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved figure -> {path.relative_to(PROJECT_ROOT)}")
    return path

In [ ]:
import importlib.metadata as importlib_metadata

def library_versions(packages=("numpy", "pandas", "scikit-learn", "xgboost", "matplotlib", "seaborn", "scipy", "shap", "joblib")) -> dict:
    versions = {"python": sys.version.split()[0], "platform": platform.platform()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not installed"
    return versions

LIBRARY_VERSIONS = library_versions()
pd.Series(LIBRARY_VERSIONS, name="version").to_frame()

## 2. Load Train and Dev features (contract-verified)
Inputs are hash-checked against `feature_spec.json`. The held-out evaluation split is
deliberately not loaded here.

In [ ]:
# Load the shared feature contract and ONLY the Train and Dev feature files.
# The held-out split is intentionally never referenced in model notebooks
# (the validation gate in notebook 10 scans this notebook's source to prove it).
FEATURE_SPEC = load_json(PATHS["artifacts"] / "feature_spec.json")

TRAIN_PATH = PATHS["features"] / "train.csv"
DEV_PATH = PATHS["features"] / "dev.csv"
DATA_FILES_READ = [str(TRAIN_PATH.relative_to(PROJECT_ROOT)), str(DEV_PATH.relative_to(PROJECT_ROOT))]

INPUT_HASHES = {"train": sha256_file(TRAIN_PATH), "dev": sha256_file(DEV_PATH)}
for split_name, digest in INPUT_HASHES.items():
    expected = FEATURE_SPEC["file_hashes"][split_name]
    assert digest == expected, (
        f"{split_name} feature file hash mismatch - re-run notebook 04 before training models."
    )

train_df = read_frame(TRAIN_PATH)
dev_df = read_frame(DEV_PATH)

X_train = train_df.drop(columns=[TARGET, ID_COLUMN])
y_train = train_df[TARGET].astype(int)
X_dev = dev_df.drop(columns=[TARGET, ID_COLUMN])
y_dev = dev_df[TARGET].astype(int)
dev_ids = dev_df[ID_COLUMN]

assert not set(LEAKAGE_COLUMNS) & set(X_train.columns), "Leakage columns reached the model inputs."
print(f"Train: {X_train.shape}, positive rate {y_train.mean():.4f}")
print(f"Dev:   {X_dev.shape}, positive rate {y_dev.mean():.4f}")

### 2.1 Shared model-notebook helpers
These do the same job in notebooks 05–09: the CV splitter, multi-metric scoring
(refit on PR-AUC), fold-score extraction and `persist_model_outputs`, which writes the
model card contract consumed by the validation gate and the evaluation notebooks.

In [ ]:
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
SCALE_POS_WEIGHT = float((y_train == 0).sum() / (y_train == 1).sum())


def build_scoring(include_probability_metrics: bool = True) -> dict:
    """Multi-metric scoring used by every model notebook (refit metric = PR-AUC)."""
    scoring = {
        "pr_auc": "average_precision",
        "roc_auc": "roc_auc",
        "f1": make_scorer(f1_score, zero_division=0),
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": make_scorer(recall_score, zero_division=0),
        "balanced_accuracy": "balanced_accuracy",
        "accuracy": "accuracy",
    }
    if include_probability_metrics:
        scoring["neg_brier"] = "neg_brier_score"
    return scoring


def load_preprocessor(family: str, variant: str):
    """Load an UNFITTED preprocessing template produced by notebook 04."""
    path = PATHS["preprocessors"] / f"preprocessor__{family}__{variant}.joblib"
    return clone(joblib.load(path))


def make_pipeline(estimator, family: str, variant: str) -> Pipeline:
    return Pipeline([("preprocess", load_preprocessor(family, variant)), ("model", estimator)])


def fold_scores_from_search(search, n_splits: int) -> pd.DataFrame:
    """Per-fold CV scores of the best candidate, read from cv_results_ (no refit needed)."""
    results, best = search.cv_results_, search.best_index_
    metrics = sorted({key.split("_test_", 1)[1] for key in results if key.startswith("split0_test_")})
    rows = []
    for fold in range(n_splits):
        row = {"fold": fold}
        for metric in metrics:
            row[metric] = results[f"split{fold}_test_{metric}"][best]
            train_key = f"split{fold}_train_{metric}"
            if train_key in results:
                row[f"train_{metric}"] = results[train_key][best]
        rows.append(row)
    return tidy_fold_scores(pd.DataFrame(rows))


def fold_scores_from_cross_validate(cv_output: dict) -> pd.DataFrame:
    frame = pd.DataFrame({
        key.replace("test_", "", 1) if key.startswith("test_") else key: values
        for key, values in cv_output.items() if key.startswith(("test_", "train_"))
    })
    frame.insert(0, "fold", range(len(frame)))
    return tidy_fold_scores(frame)


def tidy_fold_scores(frame: pd.DataFrame) -> pd.DataFrame:
    for prefix in ("", "train_"):
        if f"{prefix}neg_brier" in frame:
            frame[f"{prefix}brier"] = -frame.pop(f"{prefix}neg_brier")
    return frame


def summarise_folds(fold_scores: pd.DataFrame) -> dict:
    metric_cols = [c for c in fold_scores.columns if c != "fold" and not c.startswith("train_")]
    return {m: {"mean": float(fold_scores[m].mean()), "std": float(fold_scores[m].std(ddof=1))} for m in metric_cols}


def quick_dev_metrics(y_true, y_proba) -> dict:
    """Lightweight sanity metrics only - the official evaluation happens in notebook 11."""
    return {
        "pr_auc": float(average_precision_score(y_true, y_proba)),
        "roc_auc": float(roc_auc_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


def persist_model_outputs(*, model_key, variant, family, pipeline, fold_scores, dev_proba,
                          best_params, imbalance_strategy, fit_seconds, search=None,
                          cv_rows=None, extra=None) -> dict:
    """Write the model + the standard model-card contract consumed by notebooks 10-14."""
    model_id = f"{model_key}__{variant}"
    out_dir = PATHS["model_outputs"] / model_id
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = PATHS["models"] / f"{model_id}.joblib"
    joblib.dump(pipeline, model_path)

    fold_scores.to_csv(out_dir / "cv_fold_scores.csv", index=False)
    pd.DataFrame({ID_COLUMN: dev_ids.values, "y_true": y_dev.values, "y_proba": dev_proba}).to_csv(
        out_dir / "dev_predictions.csv", index=False
    )
    if search is not None:
        pd.DataFrame(search.cv_results_).drop(columns=["params"], errors="ignore").to_csv(
            out_dir / "search_results.csv", index=False
        )

    card = {
        "model_id": model_id,
        "model_key": model_key,
        "variant": variant,
        "notebook": NOTEBOOK_NAME,
        "preprocessor_family": family,
        "imbalance_strategy": imbalance_strategy,
        "cv": {
            "type": type(CV).__name__,
            "n_splits": CV.get_n_splits(),
            "shuffle": bool(CV.shuffle),
            "random_state": CV.random_state,
            "n_rows": int(cv_rows if cv_rows is not None else len(X_train)),
        },
        "n_fit_rows": int(len(X_train)),
        "n_dev_rows": int(len(X_dev)),
        "data_files_read": DATA_FILES_READ,
        "input_hashes": INPUT_HASHES,
        "best_params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) for k, v in best_params.items()},
        "cv_summary": summarise_folds(fold_scores),
        "dev_quick": quick_dev_metrics(y_dev, dev_proba),
        "fit_seconds": round(float(fit_seconds), 2),
        "model_path": str(model_path.relative_to(PROJECT_ROOT)),
        "model_sha256": sha256_file(model_path),
        "random_state": RANDOM_STATE,
        "library_versions": LIBRARY_VERSIONS,
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **(extra or {}),
    }
    save_json(card, out_dir / "model_card.json")
    print(f"[{model_id}] CV PR-AUC {card['cv_summary']['pr_auc']['mean']:.4f} "
          f"± {card['cv_summary']['pr_auc']['std']:.4f} | Dev PR-AUC {card['dev_quick']['pr_auc']:.4f}")
    return card

## 3. Grid search per variant
9 depths × 4 leaf sizes × 2 criteria = 72 candidates × 5 folds. Trees fit in seconds,
so an exhaustive grid is affordable and easier to justify than random search.

In [ ]:
tree_searches, tree_cards = {}, {}
for variant in VARIANTS:
    pipeline = make_pipeline(DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE), "tree", variant)
    search = GridSearchCV(pipeline, TREE_GRID, scoring=build_scoring(), refit="pr_auc", cv=CV,
                          n_jobs=-1, return_train_score=True)
    start = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    print(f"[{variant}] best params {search.best_params_}")
    tree_searches[variant] = search
    tree_cards[variant] = persist_model_outputs(
        model_key="decision_tree", variant=variant, family="tree", pipeline=search.best_estimator_,
        fold_scores=fold_scores_from_search(search, CV_FOLDS),
        dev_proba=search.best_estimator_.predict_proba(X_dev)[:, 1],
        best_params=search.best_params_, imbalance_strategy="class_weight", fit_seconds=elapsed, search=search,
        extra={"tree_depth": int(search.best_estimator_.named_steps["model"].get_depth()),
               "n_leaves": int(search.best_estimator_.named_steps["model"].get_n_leaves())},
    )

## 4. Over-fitting diagnosis: depth vs train / CV PR-AUC
For each depth, the best leaf-size/criterion combination is plotted. Train and CV
curves diverge as depth grows: this is the bias–variance trade-off that justifies a
depth limit.

In [ ]:
results = pd.DataFrame(tree_searches["day_zero"].cv_results_)
results["depth"] = results["param_model__max_depth"].astype(object).where(results["param_model__max_depth"].notna(), "None").astype(str)
depth_order = [str(d) for d in TREE_GRID["model__max_depth"]]
best_per_depth = results.loc[results.groupby("depth")["mean_test_pr_auc"].idxmax()].set_index("depth").reindex(depth_order)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(depth_order, best_per_depth["mean_train_pr_auc"], marker="o", markersize=7, color=SERIES_COLORS[0], label="Train folds")
ax.plot(depth_order, best_per_depth["mean_test_pr_auc"], marker="o", markersize=7, color=SERIES_COLORS[1], label="Validation folds")
ax.set_xlabel("max_depth")
ax.set_ylabel("PR-AUC")
ax.set_title("Deeper trees memorise Train; validation performance plateaus", loc="left")
ax.legend()
save_figure(fig, "depth_overfitting_curve")
plt.show()
best_per_depth[["mean_train_pr_auc", "mean_test_pr_auc", "std_test_pr_auc"]].round(4)

## 5. Feature importance (tuned day-zero tree)
Impurity-based importance is fast but biased toward features with many split points.
The permutation importance in notebooks 07 and 13 is the more reliable comparison.

In [ ]:
best_tree_pipeline = tree_searches["day_zero"].best_estimator_
importances = pd.Series(
    best_tree_pipeline.named_steps["model"].feature_importances_,
    index=best_tree_pipeline.named_steps["preprocess"].get_feature_names_out(),
).sort_values(ascending=False)
importances.to_csv(PATHS["tables"] / "06_decision_tree_day_zero_importances.csv", header=["importance"])

top = importances.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.index, top.values, color=COLORS["not_canceled"], height=0.65)
ax.set_xlabel("Impurity-based importance")
ax.set_title("Decision Tree (day-zero): top 15 features", loc="left")
ax.grid(axis="y", visible=False)
save_figure(fig, "importances")
plt.show()

## 6. Communication artifact: a shallow "explainer" tree
### 6.1 Fit a depth-3 tree on the day-zero features
It is deliberately *not* the tuned model and is not logged to the comparison table.
Its only purpose is human-readable rules for stakeholders.

In [ ]:
explainer_tree = make_pipeline(
    DecisionTreeClassifier(max_depth=EXPLAINER_DEPTH, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE),
    "tree", "day_zero",
).fit(X_train, y_train)
explainer_names = list(explainer_tree.named_steps["preprocess"].get_feature_names_out())
print(f"Explainer tree Dev PR-AUC: {average_precision_score(y_dev, explainer_tree.predict_proba(X_dev)[:, 1]):.4f}")

### 6.2 Rules as text

In [ ]:
rules = export_text(explainer_tree.named_steps["model"], feature_names=explainer_names, show_weights=False, decimals=2)
(PATHS["tables"] / "06_explainer_tree_rules.txt").write_text(rules)
print(rules)

### 6.3 Rules as a diagram
Node colour intensity shows class purity (orange = mostly cancelled).

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(explainer_tree.named_steps["model"], feature_names=explainer_names, class_names=["Not cancelled", "Cancelled"],
          filled=True, impurity=False, proportion=True, rounded=True, fontsize=8, ax=ax)
ax.set_title("Depth-3 explainer tree (day-zero features, class-balanced)", loc="left")
save_figure(fig, "explainer_tree")
plt.show()

## 7. Hand-off summary
- `decision_tree__day_zero` and `decision_tree__full_lifecycle` are logged to `artifacts/model_outputs/`.
- The explainer-tree rules (`reports/tables/06_explainer_tree_rules.txt`) are a ready-made stakeholder
  communication artifact for the final report.